# Multi-Factor Combination

This notebook demonstrates how to **combine multiple factors** into a single trading signal:

1. Build individual factors (momentum, volatility, volume)
2. Analyze factor correlations
3. Orthogonalize factors to remove redundancy
4. Combine factors using `CompositeFactor`
5. Compare single-factor vs. composite-factor performance
6. Factor selection workflow

**Prerequisites**: `pip install factorium`

## 1. Setup & Data Loading

In [ ]:
from factorium import BinanceDataLoader, ResearchSession
from factorium.factors import FactorAnalyzer, CompositeFactor

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "AVAXUSDT", "DOTUSDT", "CAKEUSDT",
]

loader = BinanceDataLoader()

agg = loader.load_aggbar(
    symbols=SYMBOLS,
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=30,
    bar_type="time",
    interval=60_000,
)

session = ResearchSession(agg, default_frequency="1min")
print(f"Loaded {len(agg):,} bars for {len(agg.symbols)} symbols")

## 2. Building Individual Factors

We'll create three conceptually different factors to combine.

In [ ]:
close = agg["close"]
volume = agg["volume"]

# --- Factor 1: Momentum (60-period return, ranked) ---
momentum = (close.ts_delta(60) / close.ts_shift(60)).cs_rank()
momentum.name = "momentum"

# --- Factor 2: Volatility (inverse of rolling std, ranked) ---
# Low volatility tends to outperform ("low vol anomaly")
returns = close.ts_delta(1) / close.ts_shift(1)
inv_vol = -returns.ts_std(60)  # negate so lower vol → higher signal
inv_vol_ranked = inv_vol.cs_rank()
inv_vol_ranked.name = "inv_volatility"

# --- Factor 3: Volume Ratio (relative volume, ranked) ---
# High relative volume can indicate conviction
vol_ratio = volume / volume.ts_mean(60)
vol_ratio_ranked = vol_ratio.cs_rank()
vol_ratio_ranked.name = "volume_ratio"

factors = {
    "Momentum": momentum,
    "Inv Volatility": inv_vol_ranked,
    "Volume Ratio": vol_ratio_ranked,
}

print("Individual factors created:")
for name, f in factors.items():
    print(f"  {name}: {len(f):,} rows")

### 2.1 Individual Factor IC Analysis

In [ ]:
# Compute IC for each factor
ic_table = []
for name, factor in factors.items():
    analysis = session.analyze(factor, periods=1)
    ic_stats = analysis.ic_summary.get(1, {})
    ic_table.append({
        "Factor": name,
        "Mean IC": ic_stats.get("mean_ic", np.nan),
        "IC Std": ic_stats.get("ic_std", np.nan),
        "IC IR": ic_stats.get("ic_ir", np.nan),
    })

ic_df = pd.DataFrame(ic_table).set_index("Factor")
print("Individual Factor IC Analysis:")
ic_df

## 3. Factor Correlation Analysis

Before combining factors, we need to understand how correlated they are. Highly correlated factors add little diversification benefit.

In [ ]:
# Build a merged DataFrame with all factor values aligned by time and symbol
factor_dfs = []
for name, factor in factors.items():
    df = factor.to_pandas()[["start_time", "symbol", "factor"]].rename(
        columns={"factor": name}
    )
    factor_dfs.append(df)

# Merge all factors on (start_time, symbol)
merged = factor_dfs[0]
for df in factor_dfs[1:]:
    merged = merged.merge(df, on=["start_time", "symbol"], how="inner")

# Compute correlation matrix
factor_cols = list(factors.keys())
corr_matrix = merged[factor_cols].corr()

print("Factor Correlation Matrix:")
corr_matrix

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Factor Correlation Matrix")
plt.tight_layout()
plt.show()

### 3.1 Rolling Cross-Sectional Correlation

Use `ts_corr()` to compute the time-varying correlation between two factors.

In [ ]:
# Rolling correlation between momentum and inverse volatility
rolling_corr = momentum.ts_corr(inv_vol_ranked, window=120)
rolling_corr.name = "mom_vol_corr"

rolling_corr.plot.plot_timeseries(symbols=["BTCUSDT", "ETHUSDT"])
plt.axhline(y=0, color="red", linestyle="--", alpha=0.5)
plt.title("Rolling Correlation: Momentum vs. Inverse Volatility (120-period window)")
plt.ylabel("Correlation")
plt.show()

## 4. Factor Orthogonalization

Use `cs_neutralize()` to remove the effect of one factor from another. This is useful when you want a factor that captures **unique** information.

In [ ]:
# Neutralize inverse volatility from momentum
# This gives us the "momentum" signal that is orthogonal to volatility
mom_ortho = momentum.cs_neutralize(inv_vol_ranked)
mom_ortho.name = "momentum_orthogonalized"

# Check IC of orthogonalized factor
ortho_analysis = session.analyze(mom_ortho, periods=1)
ortho_ic = ortho_analysis.ic_summary.get(1, {})

print("Orthogonalized Momentum IC:")
print(f"  Mean IC: {ortho_ic.get('mean_ic', 0):.4f}")
print(f"  IC IR:   {ortho_ic.get('ic_ir', 0):.4f}")
print()
print("Compare with original Momentum:")
print(f"  Mean IC: {ic_df.loc['Momentum', 'Mean IC']:.4f}")
print(f"  IC IR:   {ic_df.loc['Momentum', 'IC IR']:.4f}")

## 5. Combining Factors with CompositeFactor

`CompositeFactor` provides three ways to combine factors:

| Method | Description |
|--------|-------------|
| `from_equal_weights()` | Equal weight to all factors |
| `from_weights()` | Custom weights (must sum to 1) |
| `from_zscore()` | Z-score standardize each factor first, then equal weight |

In [ ]:
factor_list = [momentum, inv_vol_ranked, vol_ratio_ranked]

# --- Method 1: Equal Weights ---
composite_equal = CompositeFactor.from_equal_weights(
    factor_list, name="equal_weight"
).to_factor()

# --- Method 2: Custom Weights ---
# Give more weight to momentum, less to volume
composite_custom = CompositeFactor.from_weights(
    factor_list, weights=[0.5, 0.3, 0.2], name="custom_weight"
).to_factor()

# --- Method 3: Z-Score Standardization ---
composite_zscore = CompositeFactor.from_zscore(
    factor_list, name="zscore_combo"
).to_factor()

composites = {
    "Equal Weight": composite_equal,
    "Custom Weight (0.5/0.3/0.2)": composite_custom,
    "Z-Score Combo": composite_zscore,
}

print("Composite factors created:")
for name, f in composites.items():
    print(f"  {name}: {len(f):,} rows")

### 5.1 Composite Factor IC Analysis

In [ ]:
# Compare IC of all factors (individual + composite)
all_factors = {**factors, **composites}

full_comparison = []
for name, factor in all_factors.items():
    # Rank the composite factors for fair comparison
    signal = factor.cs_rank() if name in composites else factor
    analysis = session.analyze(signal, periods=1)
    ic_stats = analysis.ic_summary.get(1, {})
    full_comparison.append({
        "Factor": name,
        "Mean IC": ic_stats.get("mean_ic", np.nan),
        "IC Std": ic_stats.get("ic_std", np.nan),
        "IC IR": ic_stats.get("ic_ir", np.nan),
        "Type": "Composite" if name in composites else "Single",
    })

full_df = pd.DataFrame(full_comparison).set_index("Factor")
print("Full Factor IC Comparison:")
full_df

In [ ]:
# Visualize: IC IR comparison
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["steelblue" if t == "Single" else "coral" for t in full_df["Type"]]
full_df["IC IR"].plot.barh(ax=ax, color=colors)
ax.axvline(x=0, color="gray", linestyle="--")
ax.set_title("IC IR Comparison: Single vs. Composite Factors")
ax.set_xlabel("IC IR")

# Add legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="steelblue", label="Single Factor"),
    Patch(color="coral", label="Composite Factor"),
])

plt.tight_layout()
plt.show()

## 6. Backtesting: Single vs. Composite

Let's see if combining factors actually improves backtest performance.

In [ ]:
# Backtest best individual factor and best composite
bt_results = {}

for name, factor in all_factors.items():
    signal = factor.cs_rank() if name in composites else factor
    result = session.backtest(signal, neutralization="market")
    bt_results[name] = result

# Collect metrics
metrics_rows = []
for name, result in bt_results.items():
    m = result.metrics
    metrics_rows.append({
        "Factor": name,
        "Total Return": m.get("total_return", np.nan),
        "Sharpe": m.get("sharpe_ratio", np.nan),
        "Max DD": m.get("max_drawdown", np.nan),
        "Sortino": m.get("sortino_ratio", np.nan),
        "Type": "Composite" if name in composites else "Single",
    })

bt_df = pd.DataFrame(metrics_rows).set_index("Factor")
bt_df

In [ ]:
# Plot equity curves for all factors
fig, ax = plt.subplots(figsize=(14, 7))

for name, result in bt_results.items():
    eq = result.equity_curve.to_pandas()
    eq["ts"] = pd.to_datetime(eq["start_time"], unit="ms")
    style = "--" if name in factors else "-"
    linewidth = 1.0 if name in factors else 2.0
    ax.plot(eq["ts"], eq["equity"], style, label=name, linewidth=linewidth)

ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Equity Curves — Single Factors (dashed) vs. Composite Factors (solid)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Factor Selection Workflow

A practical approach to selecting which factors to include in a composite:

1. **Filter by IC**: Keep factors with |Mean IC| above a threshold
2. **Filter by correlation**: Remove highly correlated factors (keep the one with better IC)
3. **Combine remaining factors**

In [ ]:
# Step 1: Filter by absolute Mean IC
IC_THRESHOLD = 0.005  # Adjust based on your domain

selected = ic_df[ic_df["Mean IC"].abs() >= IC_THRESHOLD]
print(f"Step 1 — Factors with |Mean IC| >= {IC_THRESHOLD}:")
print(selected)
print()

In [ ]:
# Step 2: Check correlation among selected factors
CORR_THRESHOLD = 0.7  # Max acceptable pairwise correlation

selected_factors_list = [factors[name] for name in selected.index if name in factors]
selected_names = [f.name for f in selected_factors_list]

print(f"Selected factors for combination: {selected_names}")
print(f"Pairwise correlations (threshold = {CORR_THRESHOLD}):")

# Show relevant subset of correlation matrix
relevant_corr = corr_matrix.loc[
    [n for n in selected.index if n in corr_matrix.index],
    [n for n in selected.index if n in corr_matrix.columns],
]
relevant_corr

In [ ]:
# Step 3: Combine selected factors
if len(selected_factors_list) > 1:
    final_composite = CompositeFactor.from_equal_weights(
        selected_factors_list, name="selected_composite"
    ).to_factor().cs_rank()
    final_composite.name = "selected_composite"

    # Evaluate
    final_report = session.quick_report(final_composite)
    print(final_report)
else:
    print("Only one factor selected — no combination needed.")
    final_report = session.quick_report(selected_factors_list[0])
    print(final_report)

## Summary

In this notebook we:
- Built three individual factors: **momentum**, **inverse volatility**, and **volume ratio**
- Analyzed **factor correlations** using static correlation matrices and rolling `ts_corr()`
- **Orthogonalized** factors using `cs_neutralize()` to extract unique signals
- Combined factors using **`CompositeFactor`** (equal weight, custom weight, z-score)
- Compared **single-factor vs. composite-factor** performance via IC and backtesting
- Demonstrated a practical **factor selection workflow** (IC filter → correlation filter → combine)

**Key takeaway:** Combining uncorrelated factors with positive IC tends to improve IC IR and reduce portfolio volatility, even if individual IC values are small.